# main.py
import asyncio
from agent_factory.dynamic_agent_factory import DynamicAgentFactory
from config.settings import AGENT_CONFIG

async def main():
    factory = DynamicAgentFactory(AGENT_CONFIG)

    print("\n🤖 Dynamic Agent System Started")
    print("Available agents:", ", ".join(AGENT_CONFIG.keys()))
    print("Type 'switch' to change agent or 'exit' to quit.\n")

    current_agent_name = input("Enter agent name to activate: ").strip()

    while True:
        # Exit or switch logic
        if current_agent_name.lower() in ["exit", "quit"]:
            print("👋 Exiting. Goodbye!")
            break

        try:
            # Create or get agent
            agent = await factory.get_agent(current_agent_name)
            query = input(f"[{current_agent_name}] > ").strip()

            if query.lower() in ["exit", "quit"]:
                print("👋 Exiting. Goodbye!")
                break

            elif query.lower() == "switch":
                print("Available agents:", ", ".join(AGENT_CONFIG.keys()))
                current_agent_name = input("Enter agent name to switch to: ").strip()
                continue

            # Run the query
            print("\n🤔 Thinking...\n")
            result = agent.run(query)
            # result = await agent.invoke(query)
            print(f"🧠 Agent Response:\n{result}\n")

        except ValueError as e:
            print(f"❌ {e}")
            current_agent_name = input("Enter valid agent name: ").strip()
        except Exception as e:
            print(f"⚠️ Unexpected error: {e}")
            continue


if __name__ == "__main__":
    asyncio.run(main())

# dynamic_agent_factory.py
import importlib
from typing import Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool, AgentType
from mcp_clients.universal_mcp_client import load_all_mcp_tools


class DynamicAgentFactory:
    """
    Dynamically creates and manages LangChain/LangGraph agents
    based on a provided configuration.
    """

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.registry = {}

    async def create_agent(self, name: str):
        """
        Create and register an agent dynamically based on config.
        """
        if name not in self.config:
            raise ValueError(f"Agent '{name}' not found in configuration.")

        agent_cfg = self.config[name]
        print(f"🔧 Creating agent: {name}")
        print(f"   ↳ LLM: {agent_cfg['llm_model']}")
        print(f"   ↳ MCP Servers: {agent_cfg['mcp_servers']}")

        # 1️⃣ Load LLM
        llm = ChatOpenAI(model=agent_cfg["llm_model"], temperature=0.2)

        # 2️⃣ Dynamically import tools from MCP clients
        tools = []
        try:
            tools = await load_all_mcp_tools(agent_cfg)
        except Exception as e:
            print(f"⚠️ Error loading MCP tools for {name}: {e}")

        # 3️⃣ Initialize agent with proper agent type
        agent = initialize_agent(
            tools=tools,
            llm=llm,
            agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # Use enum instead of string
            verbose=True,
            handle_parsing_errors=True  # Add error handling
        )

        # 4️⃣ Store in registry
        self.registry[name] = {
            "llm": llm,
            "tools": tools,
            "agent": agent,
            "system_prompt": agent_cfg.get("system_prompt", ""),
        }

        return agent

    async def get_agent(self, name: str):
        """
        Retrieve an agent from the registry or create it if missing.
        """
        if name not in self.registry:
            await self.create_agent(name)
        return self.registry[name]["agent"]

# config/settings.py
import os
from dotenv import load_dotenv

# Load .env file from project root
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DEFAULT_MODEL = "gpt-4o"  # change to "gpt-4.1-mini" if you want cheaper


AGENT_CONFIG = {
    "agent1": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": "You are a math expert. Use math tools wisely.",
        "mcp_servers": ["math_server"]
    },
   
}        

# universal_mcp_client.py
import os
from langchain_mcp_adapters.client import MultiServerMCPClient
async def load_all_mcp_tools(agent_cfg: dict):
    """Scan for MCP servers and load all tools dynamically."""
    current_dir = os.path.dirname(os.path.abspath(__file__))
    server_path = os.path.join(current_dir, "..", "mcp_servers", "math_server.py")
    server_path = os.path.abspath(server_path)
    client=MultiServerMCPClient(
        {
            "math":{
                "command":"python",
                # "args":["mathserver.py"], ## Ensure correct absolute path
                "args":[server_path],
                "transport":"stdio",
            
            },
        }
    )
    all_tools = await client.get_tools()
    return all_tools

#math_server.py
from fastmcp import FastMCP
from typing import List, Union

mcp = FastMCP("Math Server")

@mcp.tool()
def add(numbers: List[Union[int, float]]) -> Union[int, float]:
    """Add multiple numbers together. Can handle 2, 3, or more numbers.
    
    Examples:
    - "add 2, 3, 5" -> 10
    - "sum 5 and 10 and 15" -> 30
    - "add 1, 2, 3, 4, 5" -> 15
    """
    return sum(numbers)

@mcp.tool()
def multiply(numbers: List[Union[int, float]]) -> Union[int, float]:
    """Multiply multiple numbers together. Can handle 2, 3, or more numbers.
    
    Examples:
    - "multiply 2, 3, 4" -> 24
    - "product of 5 and 10" -> 50
    - "multiply 1, 2, 3, 4" -> 24
    """
    result = 1
    for num in numbers:
        result *= num
    return result

@mcp.tool()
def subtract(numbers: List[Union[int, float]]) -> Union[int, float]:
    """Subtract numbers in sequence. First number minus the rest.
    
    Examples:
    - "subtract 10, 5" -> 5
    - "10 minus 3 minus 2" -> 5
    """
    if not numbers:
        return 0
    result = numbers[0]
    for num in numbers[1:]:
        result -= num
    return result

@mcp.tool()
def divide(numbers: List[Union[int, float]]) -> Union[int, float]:
    """Divide numbers in sequence. First number divided by the rest.
    
    Examples:
    - "divide 10, 2" -> 5
    - "20 divided by 2 divided by 5" -> 2
    """
    if not numbers:
        return 0
    result = numbers[0]
    for num in numbers[1:]:
        if num == 0:
            raise ValueError("Cannot divide by zero")
        result /= num
    return result

if __name__ == "__main__":
    print("Starting Math Server...")
    mcp.run()    